# Sightings Stay Close to the Harbors

A whale-watching trip lasts three or four hours, so it can only reach so far. If the public record is
a record of trips, sightings should crowd close to Monterey, Moss Landing and Santa Cruz. The
baseline is ship and aircraft surveys (`kind = 'survey'`), which go where their route sends them.

`km_to_harbor` is the great-circle distance to the nearest of the three, worked out in dbt, and
`dim_location.nearest_harbor` names it. The bay is the box 36.4 to 37.1 N, 122.6 to 121.7 W. A 0.1° grid
step is about 11 km, and a record on an exact grid point (`is_rounded_position`) was rounded to it, so each
group is shown with finer positions only and with all its records.

In [ ]:
%%sql
-- WHALES_WH runs the queries.
USE WAREHOUSE whales_wh

In [ ]:
%%sql
-- The star dbt builds lives in WHALES.MARTS.
USE SCHEMA whales.marts

## How Far Out Do the Records Go?
The distance within which 50, 75, 90 and 99% of each group falls.

In [ ]:
%%sql
WITH bay_records AS (
    SELECT
        dim_dataset.kind,
        fct_sighting.date_key,
        fct_sighting.km_to_harbor,
        fct_sighting.is_rounded_position
    FROM fct_sighting
    JOIN dim_location ON dim_location.location_key = fct_sighting.location_key
    JOIN dim_dataset ON dim_dataset.dataset_key = fct_sighting.dataset_key
    WHERE dim_location.in_bay
),

record_groups AS (
    SELECT 'Public, finer positions' AS record_group, date_key, km_to_harbor
    FROM bay_records
    WHERE kind = 'public' AND NOT is_rounded_position
    UNION ALL
    SELECT 'Public, all records' AS record_group, date_key, km_to_harbor
    FROM bay_records
    WHERE kind = 'public'
    UNION ALL
    SELECT 'Surveys, finer positions' AS record_group, date_key, km_to_harbor
    FROM bay_records
    WHERE kind = 'survey' AND NOT is_rounded_position
    UNION ALL
    SELECT 'Surveys, all records' AS record_group, date_key, km_to_harbor
    FROM bay_records
    WHERE kind = 'survey'
)

SELECT
    record_group,
    COUNT(*) AS records,
    COUNT(DISTINCT date_key) AS days,
    ROUND(PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY km_to_harbor), 1) AS median_km,
    ROUND(PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY km_to_harbor), 1)
        AS km_75th_percentile,
    ROUND(PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY km_to_harbor), 1)
        AS km_90th_percentile,
    ROUND(PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY km_to_harbor), 1)
        AS km_99th_percentile
FROM record_groups
GROUP BY record_group
ORDER BY median_km

## Which Harbor?
Share of public sightings by the harbor they are nearest to.

In [ ]:
%%sql
SELECT
    dim_location.nearest_harbor,
    COUNT(*) AS public_records,
    ROUND(COUNT(*) / SUM(COUNT(*)) OVER (), 3) AS share_of_public_records
FROM fct_sighting
JOIN dim_location ON dim_location.location_key = fct_sighting.location_key
JOIN dim_dataset ON dim_dataset.dataset_key = fct_sighting.dataset_key
WHERE dim_location.in_bay
  AND dim_dataset.kind = 'public'
GROUP BY dim_location.nearest_harbor
ORDER BY public_records DESC

## Result
Half of public sightings are within **10.8 km** of a harbor (10.4 km with precise positions only), and 90%
within 18.1 km. Half of ship and aircraft survey records are beyond 25.2 km (25.3 km). Moss Landing is the
harbor nearest to most public sightings, 47% of them.

## Limits
* The survey figures depend partly on how far the bay box reaches, and come from only 150 survey days.
* Moss Landing sits at the head of the Monterey submarine canyon, where whales feed, and survey routes are
  set in advance, so not all of the gap is people.
* A record on an exact 0.1° grid point was rounded and can be several km off.